# Étude de cas VIGA — 2a. Réglage du suivi

Le premier passage du suivi fragmentait beaucoup les poissons (100 à 160 pistes pour 20 poissons, médiane de 1 s) et produisait un peu plus de 20 détections par image. Ce notebook compare plusieurs réglages du suivi sur trois extraits contrastés : **T000** (le plus fragmenté), **T030** (le meilleur) et **T130** (tardif, poissons moins actifs).

Critères, sans vérité terrain :
- **nombre de pistes** (idéal : proche de 20 par extrait) ;
- **part des détections dans des pistes de 10 s et de 30 s ou plus** : le temps réellement exploitable pour calculer des indicateurs ;
- **détections par image** et **part des images avec plus de 20 détections** (doublons, fausses détections) ;
- **échanges sain ↔ malade par minute** : une piste qui passe de « marque blanche » à « pas de marque » a changé de poisson. C'est une mesure directe, mais partielle (seuls les échanges entre groupes se voient), des erreurs d'identité.

Entrées : dataset **fish-video-windows** et sortie du notebook **01_entrainement_pose**. Réglages Kaggle : **Accelerator = GPU**, Internet activé. Durée : environ 30 à 40 min.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
!pip install -q ultralytics lap

## Entrées

In [ ]:
import glob, os, shutil, time, zipfile
from pathlib import Path
import numpy as np
import pandas as pd
import cv2, torch

RACINE_ENTREE = Path(os.environ.get("RACINE_ENTREE", "/kaggle/input"))
TRAVAIL = Path(os.environ.get("TRAVAIL", "/kaggle/working"))

if not torch.cuda.is_available() and os.environ.get("AUTORISER_CPU") != "1":
    raise RuntimeError("Pas de GPU : dans le panneau de droite, Session options > Accelerator = GPU (P100 ou T4), "
                       "puis relance. Sur processeur, le calcul serait 5 à 10 fois plus long.")
DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("Calcul sur :", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "processeur")

index_trouves = glob.glob(str(RACINE_ENTREE / "**" / "index_fenetres.csv"), recursive=True)
if index_trouves:
    DOSSIER_FEN = Path(index_trouves[0]).parent
else:
    archives = glob.glob(str(RACINE_ENTREE / "**" / "fenetres*.zip"), recursive=True)
    if not archives:
        raise FileNotFoundError("Ajoute le dataset fish-video-windows en entrée (Add Input).")
    DOSSIER_FEN = TRAVAIL / "fenetres"
    with zipfile.ZipFile(archives[0]) as z:
        z.extractall(DOSSIER_FEN)
index = pd.read_csv(DOSSIER_FEN / "index_fenetres.csv")
poids = (glob.glob(str(RACINE_ENTREE / "**" / "best_yolov8n_pose.pt"), recursive=True)
         or glob.glob(str(RACINE_ENTREE / "**" / "best*.pt"), recursive=True))
if not poids:
    raise FileNotFoundError("Ajoute la sortie du notebook 01_entrainement_pose en entrée (Add Input).")
POIDS = poids[0]
print(f"{len(index)} extraits dans {DOSSIER_FEN}")
print("Poids :", POIDS)

## Réglages comparés
- **A** : réglage du premier passage. **B** : seuils de création de piste relevés (moins de pistes fantômes). **C** : B + mémoire de 5 s et association plus tolérante. **D** : C avec un seuil de détection de 0,25 (plus de détections faibles). **E** : C + ré-identification par l'apparence. **F** : ByteTrack. **G** : TrackTrack (CVPR 2025).

In [ ]:
# Réglages du suivi comparés. Tous : caméra fixe, donc pas de compensation de mouvement (gmc_method: none).
# CONF = seuil de détection transmis au tracker (les détections faibles servent à prolonger des pistes existantes).
COMMUN_BOTSORT = {"tracker_type": "botsort", "fuse_score": True, "gmc_method": "none",
                  "proximity_thresh": 0.5, "appearance_thresh": 0.8, "with_reid": False, "model": "auto"}
REGLAGES = {
    "A_actuel":            {"conf": 0.10, **COMMUN_BOTSORT, "track_high_thresh": 0.25, "track_low_thresh": 0.1,
                            "new_track_thresh": 0.25, "track_buffer": 75, "match_thresh": 0.8},
    "B_seuils_hauts":      {"conf": 0.10, **COMMUN_BOTSORT, "track_high_thresh": 0.5, "track_low_thresh": 0.1,
                            "new_track_thresh": 0.6, "track_buffer": 75, "match_thresh": 0.8},
    "C_seuils_hauts_5s":   {"conf": 0.10, **COMMUN_BOTSORT, "track_high_thresh": 0.5, "track_low_thresh": 0.1,
                            "new_track_thresh": 0.6, "track_buffer": 125, "match_thresh": 0.9},
    "D_conf_0.25_5s":      {"conf": 0.25, **COMMUN_BOTSORT, "track_high_thresh": 0.5, "track_low_thresh": 0.25,
                            "new_track_thresh": 0.6, "track_buffer": 125, "match_thresh": 0.9},
    "E_reid_5s":           {"conf": 0.10, **{**COMMUN_BOTSORT, "with_reid": True}, "track_high_thresh": 0.5,
                            "track_low_thresh": 0.1, "new_track_thresh": 0.6, "track_buffer": 125, "match_thresh": 0.9},
    "F_bytetrack_5s":      {"conf": 0.10, "tracker_type": "bytetrack", "fuse_score": True, "track_high_thresh": 0.5,
                            "track_low_thresh": 0.1, "new_track_thresh": 0.6, "track_buffer": 125, "match_thresh": 0.9},
    "G_tracktrack_5s":     {"conf": 0.10, "tracker_type": "tracktrack", "track_high_thresh": 0.6, "track_low_thresh": 0.25,
                            "new_track_thresh": 0.7, "track_buffer": 125, "match_thresh": 0.7, "lost_match_thr": 0.9,
                            "iou_weight": 0.5, "reid_weight": 0.5, "conf_weight": 0.1, "angle_weight": 0.05,
                            "penalty_p": 0.2, "penalty_q": 0.4, "reduce_step": 0.05, "tai_thr": 0.55,
                            "min_track_len": 3, "gmc_method": "none", "with_reid": False, "model": "auto"},
}

def fichier_tracker(nom):
    """Écrit le fichier de configuration du tracker et renvoie (chemin, seuil de détection)."""
    params = dict(REGLAGES[nom])
    conf = params.pop("conf")
    chemin = TRAVAIL / f"tracker_{nom}.yaml"
    chemin.write_text("".join(f"{k}: {v}\n" for k, v in params.items()))
    return str(chemin), conf

## Fonctions : marque blanche, suivi, qualité

In [ ]:
def fond_reduit(gris):
    """Fond du bassin : médiane locale (≈ 60 px) calculée sur l'image réduite 4 fois."""
    h, w = gris.shape
    petit = cv2.resize(gris, (w // 4, h // 4), interpolation=cv2.INTER_AREA)
    return cv2.medianBlur(petit, 15)

def traits_marque(gris, fond, kxy, kconf):
    """Renvoie (contraste, max_disque, fond) autour de la tête, ou NaN si les points sont peu fiables."""
    if kconf[0] < 0.5 or kconf[1] < 0.5:
        return np.nan, np.nan, np.nan
    tete, corps, queue = kxy[0], kxy[1], kxy[2]
    longueur = np.linalg.norm(tete - queue) if kconf[2] >= 0.5 else 2 * np.linalg.norm(tete - corps)
    centre = tete + 0.2 * (corps - tete)
    r = max(2.5, 0.05 * longueur)
    h, w = gris.shape
    x0, x1 = int(max(0, centre[0] - r)), int(min(w, centre[0] + r + 1))
    y0, y1 = int(max(0, centre[1] - r)), int(min(h, centre[1] + r + 1))
    if x1 - x0 < 2 or y1 - y0 < 2:
        return np.nan, np.nan, np.nan
    yy, xx = np.mgrid[y0:y1, x0:x1]
    disque = gris[y0:y1, x0:x1][np.hypot(xx - centre[0], yy - centre[1]) <= r]
    if disque.size == 0:
        return np.nan, np.nan, np.nan
    fy = int(np.clip(centre[1] // 4, 0, fond.shape[0] - 1))
    fx = int(np.clip(centre[0] // 4, 0, fond.shape[1] - 1))
    m, f = float(disque.max()), float(fond[fy, fx])
    return m - f, m, f

In [ ]:
from ultralytics import YOLO

FPS = 25
COLONNES = ["image", "id", "x1", "y1", "x2", "y2", "conf",
            "tete_x", "tete_y", "tete_c", "corps_x", "corps_y", "corps_c", "queue_x", "queue_y", "queue_c",
            "marque_contraste", "marque_max", "marque_fond"]

def suivre(modele, nom_fenetre, reglage, max_images=None, video_demo=None, duree_demo=10 * FPS):
    """Suit tous les poissons d'un extrait. Renvoie (table des détections, détections par image, images/s)."""
    tracker, conf = fichier_tracker(reglage)
    debut = time.time()
    lignes, n_det = [], []
    ecrivain, fond = None, None
    flux = modele.track(source=str(DOSSIER_FEN / f"{nom_fenetre}.mp4"), stream=True, persist=False,
                        tracker=tracker, conf=conf, iou=0.7, imgsz=640, device=DEVICE, verbose=False)
    for k, r in enumerate(flux):
        if max_images is not None and k >= max_images:
            break
        if video_demo is not None and k < duree_demo:
            img = r.plot(line_width=1, font_size=8, labels=True, conf=False)
            if ecrivain is None:
                ecrivain = cv2.VideoWriter(str(video_demo), cv2.VideoWriter_fourcc(*"mp4v"), FPS,
                                           (img.shape[1], img.shape[0]))
            ecrivain.write(img)
        if r.boxes is None or r.boxes.id is None:
            n_det.append(0); continue
        ids = r.boxes.id.int().cpu().numpy()
        boites = r.boxes.xyxy.cpu().numpy()
        confs = r.boxes.conf.cpu().numpy()
        kxy = r.keypoints.xy.cpu().numpy()
        kc = r.keypoints.conf.cpu().numpy()
        gris = cv2.cvtColor(r.orig_img, cv2.COLOR_BGR2GRAY)
        if k % FPS == 0 or fond is None:
            fond = fond_reduit(gris)
        n_det.append(len(ids))
        for j in range(len(ids)):
            c, mx, fd = traits_marque(gris, fond, kxy[j], kc[j])
            lignes.append([k, ids[j], *boites[j], confs[j],
                           kxy[j, 0, 0], kxy[j, 0, 1], kc[j, 0], kxy[j, 1, 0], kxy[j, 1, 1], kc[j, 1],
                           kxy[j, 2, 0], kxy[j, 2, 1], kc[j, 2], c, mx, fd])
    if ecrivain is not None:
        ecrivain.release()
    return pd.DataFrame(lignes, columns=COLONNES), np.array(n_det), len(n_det) / (time.time() - debut)

def qualite(df, n_det, seuil_marque=0.0, bande=10, min_marques=25):
    """Indicateurs de qualité du suivi, sans vérité terrain.
    Échanges sain <-> malade : dans une piste, passage d'une marque nette (> seuil + bande) à une absence
    nette (< seuil - bande), ou l'inverse, après lissage sur 1 s : chaque passage trahit un changement d'identité."""
    if df.empty:
        return {"n_pistes": 0}
    g = df.groupby("id")
    duree = g["image"].agg(lambda s: (s.max() - s.min() + 1) / FPS)
    taille = g.size()
    echanges, mixtes, etiquetees = 0, 0, 0
    for _, p in g:
        c = p.sort_values("image")["marque_contraste"].dropna()
        if len(c) < min_marques:
            continue
        lisse = c.rolling(FPS, min_periods=10, center=True).median().dropna().to_numpy()
        etat = np.where(lisse > seuil_marque + bande, 1, np.where(lisse < seuil_marque - bande, -1, 0))
        etat = etat[etat != 0]
        n_ech = int(np.sum(etat[1:] != etat[:-1])) if len(etat) > 1 else 0
        echanges += n_ech
        mixtes += n_ech > 0
        etiquetees += n_ech == 0 and len(etat) > 0
    minutes = len(n_det) / FPS / 60
    return {
        "n_pistes": int(len(taille)), "duree_mediane_s": float(duree.median()),
        "part_det_pistes_10s": float(taille[duree >= 10].sum() / taille.sum()),
        "part_det_pistes_30s": float(taille[duree >= 30].sum() / taille.sum()),
        "det_par_image": float(n_det.mean()), "part_images_plus_de_20": float((n_det > 20).mean()),
        "echanges_sain_malade_par_min": echanges / minutes, "pistes_mixtes": int(mixtes),
        "pistes_etiquetees_sans_echange": int(etiquetees),
    }

## Comparaison

In [ ]:
FENETRES_TEST = os.environ.get("FENETRES_TEST", "T000,T030,T130").split(",")
MAX_IMAGES = int(os.environ["MAX_IMAGES"]) if os.environ.get("MAX_IMAGES") else None
modele = YOLO(POIDS)
detail = []
for reglage in REGLAGES:
    for fen in FENETRES_TEST:
        try:
            df, n_det, ips = suivre(modele, fen, reglage, max_images=MAX_IMAGES)
            q = qualite(df, n_det)
            detail.append({"reglage": reglage, "fenetre": fen, **q, "images_par_s": ips})
            print(f"[{reglage} | {fen}] {q['n_pistes']} pistes, {q['part_det_pistes_30s']:.0%} des détections en pistes >= 30 s, "
                  f"{q['det_par_image']:.1f} détections/image, {q['echanges_sain_malade_par_min']:.2f} échanges/min, {ips:.0f} img/s")
        except Exception as ex:
            detail.append({"reglage": reglage, "fenetre": fen, "erreur": str(ex)[:200]})
            print(f"[{reglage} | {fen}] ÉCHEC : {ex}")
detail = pd.DataFrame(detail)
detail.to_csv(TRAVAIL / "reglage_detail.csv", index=False)

## Synthèse et recommandation
Critère principal : **poissons couverts par des pistes de 30 s ou plus, par image** (= détections par image × part en pistes ≥ 30 s). Un réglage qui supprime des détections ne peut donc pas paraître meilleur. Parmi les réglages qui ne font pas plus d'échanges sain ↔ malade que le réglage actuel (marge de 10 %), on retient celui qui couvre le plus de poissons.

In [ ]:
ok = detail[detail["erreur"].isna()] if "erreur" in detail.columns else detail
resume = ok.groupby("reglage").agg(
    pistes_par_extrait=("n_pistes", "mean"), duree_mediane_s=("duree_mediane_s", "mean"),
    part_det_pistes_10s=("part_det_pistes_10s", "mean"), part_det_pistes_30s=("part_det_pistes_30s", "mean"),
    det_par_image=("det_par_image", "mean"), part_images_plus_de_20=("part_images_plus_de_20", "mean"),
    echanges_par_min=("echanges_sain_malade_par_min", "mean"), images_par_s=("images_par_s", "mean"),
).reindex([r for r in REGLAGES if r in set(ok["reglage"])])
resume.insert(0, "poissons_couverts_30s", resume["det_par_image"] * resume["part_det_pistes_30s"])
resume.to_csv(TRAVAIL / "reglage_resume.csv")
print(resume.round(3).to_string())

reference = resume.loc["A_actuel", "echanges_par_min"] if "A_actuel" in resume.index else resume["echanges_par_min"].min()
admissibles = resume[resume["echanges_par_min"] <= reference * 1.1 + 0.05]
CHOIX = admissibles["poissons_couverts_30s"].idxmax() if len(admissibles) else resume["poissons_couverts_30s"].idxmax()
print(f"\nRéglage recommandé : {CHOIX}  ->  à reporter dans le notebook 02_suivi (variable CHOIX)")
(TRAVAIL / "reglage_retenu.txt").write_text(CHOIX + "\n")

In [ ]:
import matplotlib.pyplot as plt
FOND, ENCRE, ENCRE_2, GRILLE, SERIE = "#fcfcfb", "#0b0b0b", "#52514e", "#e4e3df", "#2a78d6"
fig, axes = plt.subplots(1, 3, figsize=(13, 0.45 * len(resume) + 1.6), sharey=True)
mesures = [("poissons_couverts_30s", "Poissons par image couverts par des pistes ≥ 30 s", False),
           ("echanges_par_min", "Échanges sain ↔ malade par minute", False),
           ("pistes_par_extrait", "Pistes par extrait (20 poissons)", False)]
for ax, (col, titre, pourcent) in zip(axes, mesures):
    ax.set_facecolor(FOND)
    y = np.arange(len(resume))
    couleurs = [ENCRE if r == CHOIX else SERIE for r in resume.index]
    ax.barh(y, resume[col], color=couleurs, height=0.6)
    for yi, v in zip(y, resume[col]):
        ax.text(v, yi, f" {v:.0%}" if pourcent else f" {v:.1f}", va="center", fontsize=8, color=ENCRE_2)
    ax.set_title(titre, loc="left", fontsize=9)
    ax.set_yticks(y, resume.index, fontsize=8)
    ax.set_xlim(left=0)
    ax.invert_yaxis()
    for s in ["top", "right"]:
        ax.spines[s].set_visible(False)
    ax.grid(axis="x", color=GRILLE); ax.set_axisbelow(True)
    ax.tick_params(colors=ENCRE_2, labelsize=8)
axes[2].axvline(20, color=ENCRE_2, linewidth=1, linestyle="--")
fig.suptitle(f"Réglage du suivi sur {', '.join(FENETRES_TEST)} (en noir : réglage retenu)", x=0.01, ha="left", fontsize=10)
fig.patch.set_facecolor(FOND)
plt.tight_layout(); plt.savefig(TRAVAIL / "reglage_suivi.png", dpi=150); plt.show()